# LLM Experimentation Notebook
### Tokenization, Embeddings, Semantic Search & API Parameter Experiments

**Instructions:**
- Sections 1–2 (Tokenization, Embeddings) run fully offline using open-source libraries — no API key needed.
- Section 3 (LLM API experimentation) requires an API key (Groq). Instructions are provided; skip/mock if you don't have one, and use a Playground UI instead (see exercises sheet).
- Fill in the `# TODO` cells yourself, run each cell, and write a short observation in the markdown cell provided after each exercise.


## Import Required Libraries

In [6]:
import configparser
import os
import tiktoken
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
from jinja2 import Template

load_dotenv()

True

## Section 1 — Tokenization

**Goal:** See how text gets broken into tokens, and how token count varies by content and model.


In [4]:
# GPT-style tokenizer (cl100k_base is used by GPT-3.5/4 family; good general-purpose example)
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # tokenizer is created

sample_sentences = [
    "When your had a bad day and the weather makes your mood happy",
    "supercalifragilisticexpialidocious",
    "Masked self token prevents a token from looking in future token.",
    "नमस्ते, आप कैसे हैं?",       # non-English example
    "def add(a, b):\n    return a + b"  # code example
    # My added sentences
    "Artificial intelligence models use deep neural networks to process complex data",
    "The kids love to play outside at evening",
    "The area of a circle is calculated using the formula A = pi r^2.",  # Mathematical sentence
]

for s in sample_sentences:
    tokens = enc.encode(s)
    print(f"Text: {s!r}")
    print(f"  Token count: {len(tokens)}")
    print(f"  Tokens (decoded individually): {[enc.decode([t]) for t in tokens]}")
    print()


Text: 'When your had a bad day and the weather makes your mood happy'
  Token count: 13
  Tokens (decoded individually): ['When', ' your', ' had', ' a', ' bad', ' day', ' and', ' the', ' weather', ' makes', ' your', ' mood', ' happy']

Text: 'supercalifragilisticexpialidocious'
  Token count: 11
  Tokens (decoded individually): ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']

Text: 'Masked self token prevents a token from looking in future token.'
  Token count: 13
  Tokens (decoded individually): ['Mask', 'ed', ' self', ' token', ' prevents', ' a', ' token', ' from', ' looking', ' in', ' future', ' token', '.']

Text: 'नमस्ते, आप कैसे हैं?'
  Token count: 20
  Tokens (decoded individually): ['न', 'म', 'स', '्�', '�', 'े', ',', ' �', '�', 'प', ' क', '�', '�', 'स', 'े', ' ह', '�', '�', 'ं', '?']

Text: 'def add(a, b):\n    return a + bArtificial intelligence models use deep neural networks to process complex data'
  Token count: 23
  Tokens (decoded indivi

**Exercise 1.1:** Add 3 of your own sentences to `sample_sentences` above (try one long technical sentence, one sentence with an uncommon/made-up word, and one in a language other than English). Rerun the cell.

**Reflection (write here):** Which sentence had the most tokens relative to its word count? Why do you think that happened? _(TODO: your answer)_

Answer:
The hindi sentence (नमस्ते, आप कैसे हैं?) and the word (supercalifragilisticexpialidocious) had highest token-to-word ratio.

This happens because the tokenizer(cl100k_base)is primarily trained on standard English text.When it come across non-english scripts or rare terms so it cannot map them to one word so it breaks them down into multiple smaller sub-word chunks due to which token count to jump higher than the word count.


In [5]:
# TODO: Exercise 1.2 — Compare token count vs. word count
# For each sentence in sample_sentences, print: word_count, token_count, and the ratio (tokens/word)

for s in sample_sentences:
    tokens = enc.encode(s) #converts the sentence into token ids
    word_count = len(s.split())
    token_count = len(tokens)
    ratio = token_count / max(word_count, 1)
    print(f"{s[:40]!r:45} words={word_count:3}  tokens={token_count:3}  ratio={ratio:.2f}")


'When your had a bad day and the weather '    words= 13  tokens= 13  ratio=1.00
'supercalifragilisticexpialidocious'          words=  1  tokens= 11  ratio=11.00
'Masked self token prevents a token from '    words= 11  tokens= 13  ratio=1.18
'नमस्ते, आप कैसे हैं?'                        words=  4  tokens= 20  ratio=5.00
'def add(a, b):\n    return a + bArtificia'   words= 17  tokens= 23  ratio=1.35
'The kids love to play outside at evening'    words=  8  tokens=  8  ratio=1.00
'The area of a circle is calculated using'    words= 14  tokens= 17  ratio=1.21


## Section 2 — Embeddings & Semantic Search

**Goal:** Generate embeddings for sentences and use cosine similarity to find semantically related sentences — the intuition behind semantic search (Week 2 preview).


In [7]:
# A small, fast open-source embedding model — good for classroom use
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "I love hiking in the mountains.",
    "The stock market fell sharply today.",
    "My dog loves to play fetch in the park.",
    "Interest rates rose this quarter.",
    "We went camping last weekend near the lake."
]

embeddings = model.encode(sentences)
print("Embedding shape per sentence:", embeddings.shape)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3038.94it/s]


Embedding shape per sentence: (5, 384)


In [ ]:
# embeddings[0]

In [8]:
# Compute pairwise cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix:\n")
print("      " + "  ".join([f"S{i+1}" for i in range(len(sentences))]))
for i, row in enumerate(sim_matrix):
    print(f"S{i+1}  " + "  ".join([f"{v:.2f}" for v in row]))

print()
for i, s in enumerate(sentences):
    print(f"S{i+1}: {s}")


Cosine Similarity Matrix:

      S1  S2  S3  S4  S5
S1  1.00  -0.07  0.23  0.08  0.36
S2  -0.07  1.00  -0.01  0.28  -0.04
S3  0.23  -0.01  1.00  0.04  0.25
S4  0.08  0.28  0.04  1.00  0.05
S5  0.36  -0.04  0.25  0.05  1.00

S1: I love hiking in the mountains.
S2: The stock market fell sharply today.
S3: My dog loves to play fetch in the park.
S4: Interest rates rose this quarter.
S5: We went camping last weekend near the lake.


**Exercise 2.1 (Part H from exercises sheet):** Look at the similarity matrix above.
- Which pair of sentences has the highest similarity (excluding a sentence with itself)?
Answer:Highest Similarity Pair: Sentences S1 ("I love hiking in the mountains.") and S5 ("We went camping last weekend near the lake.") have the highest similarity score (0.36).

- Does this match your intuition? _(TODO: your answer)_
Answer: Yes , this matches human intuition.

**Exercise 2.2:** Add 2 new sentences of your own — one that should be semantically close to an existing sentence, and one that should be far from all of them. Rerun Section 2 and confirm your prediction.


In [9]:
# TODO: Exercise 2.2 — add your own sentences and rerun
my_sentences = sentences + [
    "I enjoy walking through the mountain forests.",
    "I love baking homemade chocolate chip cookies."
]
print("Updated Cosine Similarity Matrix:")
my_embeddings = model.encode(my_sentences)
my_sim_matrix = cosine_similarity(my_embeddings)

for i, row in enumerate(my_sim_matrix):
    print(f"S{i+1}  " + "  ".join([f"{v:.2f}" for v in row]))


Updated Cosine Similarity Matrix:
S1  1.00  -0.07  0.23  0.08  0.36  0.78  0.33
S2  -0.07  1.00  -0.01  0.28  -0.04  -0.09  -0.02
S3  0.23  -0.01  1.00  0.04  0.25  0.25  0.22
S4  0.08  0.28  0.04  1.00  0.05  0.07  0.13
S5  0.36  -0.04  0.25  0.05  1.00  0.33  0.14
S6  0.78  -0.09  0.25  0.07  0.33  1.00  0.26
S7  0.33  -0.02  0.22  0.13  0.14  0.26  1.00


In [11]:
# TODO: Exercise 2.3 — Simple semantic search function
# Given a query sentence, find the most similar sentence from `sentences` using cosine similarity.

def semantic_search(query, corpus, corpus_embeddings, top_k=1):
    query_embedding = model.encode([query])
    sims = cosine_similarity(query_embedding, corpus_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(corpus[i], sims[i]) for i in top_idx]

# Try it out
query = "baking sweet desserts in the kitchen"  # TODO: try your own queries too
results = semantic_search(query, sentences, embeddings, top_k=3)
for text, score in results:
    print(f"{score:.3f}  |  {text}")


0.119  |  We went camping last weekend near the lake.
0.100  |  I love hiking in the mountains.
0.092  |  My dog loves to play fetch in the park.


**Reflection:** Try a query using a synonym or related concept that does NOT share exact keywords with any sentence (e.g., "financial markets" instead of "stock market"). Does semantic search still find the right sentence? Compare this to what a simple keyword search (`if word in sentence`) would have found. _(TODO: your answer)_

Answer:
Query Tested: "Baking sweet desserts in the kitchen"
Sematic Search: Successful
Comparision Keyword search: Would have given 0 result because query doesnt matches with the sentences. 

## Section 3 — LLM API Experimentation (Parameters)

**Goal:** Directly observe how `temperature`, `max_tokens`, and system prompts affect model output.

> **Note:** This section requires an API key. Set it as an environment variable before running (`GROQ_API_KEY`)and log your observations.


In [12]:
# Make sure _API_KEY is set in your environment before running this cell
hf_token = os.getenv("HF_TOKEN")

In [13]:
def load_config():
    try:
        # Create a ConfigParser object
        config = configparser.ConfigParser()
        # Read the config.ini file
        config_path = os.path.join("../../config/config.ini")
        config.read(config_path)
        return config
    except FileNotFoundError:
        raise
    except Exception:
        raise

In [14]:
config = load_config()
llm = InferenceClient(
    model=config.get("LLM_MODEL", "model"),
    token=hf_token,
    timeout=config.getint("LLM_MODEL", "hub_timeout")
)

In [15]:
def ask_hf(prompt, model, temperature=1.0, max_tokens=200):
    output = llm.chat_completion(
        messages=prompt,
        max_tokens=max_tokens,
        temperature=temperature
    )
    return output.choices[0].message.content

### Exercise 3.1 — Temperature

Run the same creative prompt at three different temperatures. Compare creativity vs. consistency (run each setting 2–3 times).


In [17]:
prompt = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain quantum computing in one sentence."}
]

for temp in [0.0, 0.7, 1.2]:
    print(f"\n--- Temperature = {temp} ---")
    for i in range(2):
        output = ask_hf(prompt=prompt, model=llm, temperature=temp, max_tokens=60)
        print(f"Run {i+1}: {output}")


--- Temperature = 0.0 ---
Run 1: Quantum computing is a revolutionary technology that uses the principles of quantum mechanics to perform calculations and operations on data at an exponentially faster rate than classical computers by leveraging the unique properties of quantum bits (qubits).
Run 2: Quantum computing is a revolutionary technology that uses the principles of quantum mechanics, such as superposition and entanglement, to perform calculations and operations on data that are exponentially faster and more powerful than those of classical computers.

--- Temperature = 0.7 ---
Run 1: Quantum computing is a revolutionary technology that uses the principles of quantum mechanics, such as superposition and entanglement, to perform calculations and operations on data that are exponentially faster and more powerful than those of classical computers.
Run 2: Quantum computing is a type of computing that uses the principles of quantum mechanics, such as superposition and entanglement, 

**Reflection:** At temperature 0, did repeated runs give (near-)identical outputs? What changed as temperature increased? _(TODO: your answer)_

Answer:
Temp 0.0 ,both runs produced nearly identical definations.


### Exercise 3.2 — Max Tokens

Ask for a detailed explanation with a very low `max_tokens` limit, then a higher one. Observe where the output gets cut off.


In [19]:
prompt = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain how transformers use self-attention, in detail."}
        ]

# TODO: uncomment once your API key is set
short_output = ask_hf(prompt=prompt, model=llm, max_tokens=20)
long_output = ask_hf(prompt=prompt, model=llm, max_tokens=300)
print("SHORT (max_tokens=20):\n", short_output)
print("\nLONG (max_tokens=300):\n", long_output)


SHORT (max_tokens=20):
 The transformer architecture, introduced in the paper "Attention is All You Need" by Vaswani et

LONG (max_tokens=300):
 A fascinating topic! Transformers, introduced in the 2017 paper "Attention is All You Need" by Vaswani et al., revolutionized the field of natural language processing (NLP) and computer vision. At the heart of this breakthrough is the concept of self-attention, which I'd be happy to explain in detail.

**What is self-attention?**

Self-attention, also known as intra-attention, is a mechanism that allows a model to selectively focus on different parts of its input while generating the output. In the context of transformers, self-attention is used to compute weighted sums of different input representations, where the weights are learned during training.

**Key components of self-attention:**

1. **Query (Q)**: A set of input representations, typically a sequence of vectors.
2. **Key (K)**: Another set of input representations, often the same as 

### Exercise 3.3 — System Prompt vs. User Prompt

Set a system-level instruction and send several different user questions to see if it's consistently followed.


In [20]:
system_instruction = "Always answer in exactly one sentence, no matter the question."

questions = [
    "What is a transformer model?",
    "Why do we need tokenization?",
    "What is the difference between pretraining and fine-tuning?",
]

for q in questions:
    # TODO: uncomment once your API key is set
    prompt = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": q}
        ]
    output = ask_hf(prompt=prompt, model=llm, max_tokens=100)
    print(f"Q: {q}\nA: {output}\n")
    pass


Q: What is a transformer model?
A: A transformer model is a type of neural network architecture that uses self-attention mechanisms to process sequential data, such as natural language or images, in parallel and simultaneously, enabling parallelization and efficiency.

Q: Why do we need tokenization?
A: Tokenization is necessary because it allows computers to analyze and process large amounts of unstructured text data, such as articles, emails, and social media posts, by breaking them down into smaller, manageable units called tokens.

Q: What is the difference between pretraining and fine-tuning?
A: Pretraining involves training a deep learning model on a large, general dataset to learn general representations, whereas fine-tuning involves adapting the pre-trained model to a specific task or dataset by adjusting its weights to fit the new data.



**Reflection:** Was the system instruction followed consistently across all three questions? Note any cases where it was ignored or only partially followed. _(TODO: your answer)_

Answer:
The system prompt successfully executed
---

## Wrap-Up

Write a 3–5 sentence summary of what you learned about tokenization, embeddings, and LLM parameters, and one open question you still have.

MY SUMMARY:
Tokenization splits text into numbers
The model can read, and embeddings turn those numbers into vectors.So the system understands meaning instead of just matching words.
Testing the parameters proved that zero temperature keeps answers consistent, max_tokens cuts off responses and system prompts force the AI to follow fixed rules.